Fine-tuning continues training a large pretrained model on a smaller dataset specific to a task or domain.

# Parameters

In [ ]:
from ipywidgets import widgets

style = {"description_width": "220px"}
layout = widgets.Layout(width="600px")

tokenizer_input = widgets.Text(
    value="/workspace/data/llms/gpt-oss-20b",
    placeholder="Enter tokenizer path...",
    description="Tokenizer path:",
    style=style,
    layout=layout,
)

model_input = widgets.Text(
    value="/workspace/data/llms/gpt-oss-20b",
    placeholder="Enter model path...",
    description="Model path:",
    style=style,
    layout=layout,
)

widgets.VBox(
    [tokenizer_input, model_input], layout=widgets.Layout(padding="12px", gap="8px")
)

In [1]:
from huggingface_hub import login

login()

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
dataset = load_dataset("karthiksagarn/astro_horoscope", split="train")


def tokenize(batch):
    return tokenizer(
        batch["horoscope"],
        truncation=True,
        max_length=512,
    )


dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
data_collator = (DataCollatorForLanguageModeling(tokenizer, mlm=False),)

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")

In [ ]:
training_args = TrainingArguments(
    output_dir="qwen3-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()
trainer.push_to_hub()